# Lab 5A — The Function *Is* the Tool

Every tool so far was a Python function you wrote and a JSON schema you typed by
hand, kept in step with each other by care and luck.

A **Unity Catalog function** replaces all three. Its signature becomes the schema,
its `COMMENT` becomes the description the model reads, and its `GRANT` becomes the
access control. One object, governed once.

| Cell group | What you do |
|---|---|
| 1 | Write a function whose `COMMENT` is written **for a model**. |
| 2 | Call it as SQL. |
| 3 | **Read back the tool schema Databricks generated from it.** |
| 4 | Grant `EXECUTE` and withhold `SELECT` — least privilege in two statements. |
| 5 | Prove the boundary holds. |

**Compute:** Serverless.

In [0]:
CATALOG, SCHEMA = "agents_labs", "retail"

# The agent's execution identity. Unity Catalog identifies a service principal by
# its APPLICATION ID, not its display name -- and display names are not unique.
from databricks.sdk import WorkspaceClient
_w = WorkspaceClient()
_sps = [sp for sp in _w.service_principals.list()
        if sp.display_name == "agents-labs-restricted"]
if not _sps:
    raise RuntimeError("no service principal named 'agents-labs-restricted'")
SP_ID = _sps[0].application_id

print(f"  catalog.schema : {CATALOG}.{SCHEMA}")
print(f"  principals named 'agents-labs-restricted': {len(_sps)}")
print(f"  granting to application_id: {SP_ID}")

  catalog.schema : agents_labs.retail
  principals named 'agents-labs-restricted': 2
  granting to application_id: a43d91d6-5848-46f7-99d6-70737fd76451


## 1. Write the function — and write the COMMENT for a model

Read the `COMMENT` below as if you were the model choosing a tool. It says what the
function returns **and when to use it**. That second half is what stops the agent
guessing.

Note the parameter comment too: it becomes the parameter description, so the model
knows an order reference looks like `ORD-1044`.

In [0]:
%sql
CREATE OR REPLACE FUNCTION agents_labs.retail.get_order_summary(
  order_ref STRING COMMENT 'Customer-facing order reference, e.g. ORD-1044'
)
RETURNS TABLE (
  order_id STRING, status STRING, item STRING, units INT,
  revenue DECIMAL(12,2), order_date DATE, region STRING, tier STRING
)
COMMENT 'Look up one order: fulfilment status, item, units, revenue, and the region and loyalty tier of the customer who placed it. Use this whenever a question refers to a specific order reference.'
RETURN
  SELECT o.order_id, o.status, o.item, o.units, o.revenue, o.order_date, c.region, c.tier
  FROM agents_labs.retail.orders o
  JOIN agents_labs.retail.customers c USING (customer_id)
  WHERE o.order_id = order_ref;

## 2. It is an ordinary SQL function

Before it is a tool, it is something an analyst can call.

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.get_order_summary('ORD-1044')"))

order_id,status,item,units,revenue,order_date,region,tier
ORD-1044,delivered,standing desk,5,2700.00,2026-09-18,Nordics,standard


## 3. Now read the tool schema Databricks generated

Databricks exposes Unity Catalog functions as **MCP tools** at
`/api/2.0/mcp/functions/{catalog}/{schema}`. Ask it what tools exist, and look at
what came back.

**You did not write any of this.** It was derived from the function.

In [0]:
import json, requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
HOST = w.config.host
MCP = f"{HOST}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"


def rpc(token: str, method: str, params: dict | None = None) -> dict:
    r = requests.post(
        MCP,
        headers={"Authorization": f"Bearer {token}",
                 "Content-Type": "application/json",
                 "Accept": "application/json, text/event-stream"},
        json={"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}},
        timeout=90)
    if r.status_code != 200:
        return {"http_error": r.status_code, "body": r.text[:300]}
    return r.json()


MY_TOKEN = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
tools = rpc(MY_TOKEN, "tools/list").get("result", {}).get("tools", [])

print(f"  tools discovered as {w.current_user.me().user_name}: {len(tools)}\n")
for t in tools:
    print(f"    {t['name']}")

spec = next((t for t in tools if t["name"].endswith("get_order_summary")), None)
print("\n  the generated schema:\n")
print(json.dumps(spec, indent=2)[:1200])

  tools discovered as admin@datacouchoutlook.onmicrosoft.com: 2

    agents_labs__retail__get_order_summary
    agents_labs__retail__revenue_by_region

  the generated schema:

{
  "name": "agents_labs__retail__get_order_summary",
  "description": "Look up one order: fulfilment status, item, units, revenue, and the region and loyalty tier of the customer who placed it. Use this whenever a question refers to a specific order reference.",
  "inputSchema": {
    "type": "object",
    "required": [
      "order_ref"
    ],
    "properties": {
      "order_ref": {
        "type": "string",
        "description": "Customer-facing order reference, e.g. ORD-1044"
      }
    }
  },
  "outputSchema": {
    "type": "object",
    "required": [
      "is_truncated"
    ],
    "properties": {
      "is_truncated": {
        "type": "boolean"
      },
      "columns": {
        "type": "array",
        "items": {
          "type": "string"
        }
      },
      "rows": {
        "type": "array",


### Where each piece came from

| In the tool schema | Came from |
|---|---|
| `name` | the fully-qualified function name |
| `description` | the function's **`COMMENT`** |
| `inputSchema.properties.order_ref` | the **parameter**, with its own `COMMENT` |
| `required` | the parameter having no default |

> 💡 **This is the argument for `COMMENT ON`.** An undocumented function produces a
> tool the model cannot choose correctly, and the fix is a `COMMENT`, not a longer
> system prompt. The comment also improves Catalog Explorer, Genie
> ([Lab 4A](#)) and every analyst who reads the schema — you write it once.

## 4. Least privilege, in two statements

The agent runs as a **service principal**, not as you. Give it exactly what it needs:
the right to *run the function*, and **no** right to read the tables underneath.

A UC function runs with **definer's rights** — it executes as its owner, so the
caller never needs `SELECT` on `orders` or `customers`.

> ⚠️ **Grant to the application ID, not the display name.** The first version of this
> cell used `TO \`agents-labs-restricted\`` and failed with
> `PRINCIPAL_DOES_NOT_EXIST: Could not find principal with name agents-labs-restricted`.
>
> Unity Catalog identifies a service principal by its **application ID** (a UUID).
> Display names are not unique — this workspace has **two** principals called
> `agents-labs-restricted`, which is exactly the situation where granting by name
> would be ambiguous even if it worked.

In [0]:
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{SP_ID}`")
spark.sql(f"GRANT USE SCHEMA  ON SCHEMA  {CATALOG}.{SCHEMA} TO `{SP_ID}`")
spark.sql(f"GRANT EXECUTE ON FUNCTION {CATALOG}.{SCHEMA}.get_order_summary TO `{SP_ID}`")

print(f"  granted USE CATALOG, USE SCHEMA, EXECUTE to {SP_ID}")
print("  deliberately NOT granted: SELECT on orders, SELECT on customers\n")

for obj, kind in ((f"{CATALOG}.{SCHEMA}.get_order_summary", "FUNCTION"),
                  (f"{CATALOG}.{SCHEMA}.orders", "TABLE")):
    rows = spark.sql(f"SHOW GRANTS ON {kind} {obj}").collect()
    mine = [r for r in rows if SP_ID in (r["Principal"] or "")]
    print(f"  {kind} {obj.split('.')[-1]:20} -> "
          f"{[r['ActionType'] for r in mine] or 'no grants to the SP'}")

  granted USE CATALOG, USE SCHEMA, EXECUTE to a43d91d6-5848-46f7-99d6-70737fd76451
  deliberately NOT granted: SELECT on orders, SELECT on customers

  FUNCTION get_order_summary    -> ['EXECUTE']
  TABLE orders               -> no grants to the SP


## 5. What you built

| Piece | Cell | Why it matters |
|---|---|---|
| The function | 1 | one object, not a function *and* a schema |
| The `COMMENT` | 1, 3 | **is** the tool description the model reads |
| The parameter `COMMENT` | 1, 3 | **is** the parameter description |
| MCP `tools/list` | 3 | the schema, generated, not typed |
| `EXECUTE` without `SELECT` | 4 | least privilege in two statements |
| Definer's rights | 4 | why that is even possible |

### Try this before you move on

1. Change the `COMMENT` to something vague — *"gets order stuff"* — re-run cells 1
   and 3, and read the new description. That is what the model would have to choose
   from.
2. Add a second parameter without a `COMMENT`. What does its description become?
3. [Lab 5B](#) calls this function as the service principal, and as you, and the two
   identities do not see the same tool list.